# Grasp adaptation — compliance-matched stiffness control

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys
from pathlib import Path

sys.path.insert(0, os.path.join('../'))
from plot_config import draw_radar

OUTPUT_DIR = os.path.join('outputs', 'grasp_adaptation')
OBJECTS    = ['hard_obj', 'soft_obj']
COLORS     = {'hard_obj': '#0072B2', 'soft_obj': '#D55E00'}

def load_latest(obj):
    folder = Path(OUTPUT_DIR)
    if not folder.exists():
        return None
    files = sorted(folder.glob(f'grasp_{obj}*.csv'))
    return pd.read_csv(files[-1]) if files else None

data = {obj: load_latest(obj) for obj in OBJECTS}

## Sensing phase — fingertip displacement per object

In [11]:
fig, ax = plt.subplots()

FINGERTIPS = ['thumb', 'index', 'middle', 'ring', 'pinky']
x = np.arange(len(FINGERTIPS))
width = 0.35

for i, (obj, color) in enumerate(COLORS.items()):
    df = data[obj]
    if df is None:
        continue
    sense = df[df['phase'] == 'sense']
    probe = df[df['phase'] == 'probe']
    deltas = []
    for f in FINGERTIPS:
        cols = [f'tip_{f}_x_m', f'tip_{f}_y_m', f'tip_{f}_z_m']
        pos_gentle = sense[cols].median().to_numpy()
        pos_probe  = probe[cols].median().to_numpy()
        deltas.append(np.linalg.norm(pos_probe - pos_gentle) * 1e3)
    ax.bar(x + i * width, deltas, width,
           label=obj.replace('_', ' ').title(), color=color)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(FINGERTIPS)
ax.set_xlabel('Finger')
ax.set_ylabel(r'$\|\Delta x_f\|$ [mm]')
ax.legend(fontsize='small')
fig.tight_layout()
os.makedirs(OUTPUT_DIR, exist_ok=True)
fig.savefig(os.path.join(OUTPUT_DIR, 'sensing_displacement.pdf'), bbox_inches='tight')
plt.show()


## Adapted stiffness — sensing displacement vs k_applied

In [12]:
fig, ax = plt.subplots()

from hand_config import K_GAIN, K_MIN, K_MAX
FINGERTIPS = ['thumb', 'index', 'middle', 'ring', 'pinky']

# Adaptation hyperbola k = K_GAIN / C_O, clipped to [K_MIN, K_MAX]
co_range = np.linspace(1e-4, 0.10, 500)
k_curve  = np.clip(K_GAIN / co_range, K_MIN, K_MAX)
ax.plot(co_range * 1e3, k_curve, color='0.5', lw=1, linestyle='--',
        label='adaptation rule')

for obj, color in COLORS.items():
    df = data[obj]
    if df is None:
        continue
    # k_applied and C_O_mean are set only after STATE_PROBE_REC — read from post-probe phases
    post = df[df['phase'].isin(['adapt', 'lift', 'hold', 'place'])]
    if post.empty:
        continue
    k_app = float(post['k_applied_Npm'].iloc[0])
    c_o   = float(post['C_O_m_per_N'].iloc[0])
    ax.scatter([c_o * 1e3], [k_app], color=color, zorder=5,
               label=obj.replace('_', ' ').title())
    ax.annotate(f'{k_app:.0f} N/m', (c_o * 1e3, k_app),
                textcoords='offset points', xytext=(6, 4))

ax.set_xlabel(r'$C_O$ [mm/N]')
ax.set_ylabel(r'$k_\mathrm{applied}$ [N/m]')
ax.legend(fontsize='small')
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'adaptation_curve.pdf'), bbox_inches='tight')
plt.show()
